You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: 

Title: Unraveling the World of Artificial Intelligence: Trends, Players, and News

Introduction:
- Brief overview of Artificial Intelligence (AI) and its impact on various industries
- Importance of staying updated on the latest trends and news in the AI field

Key Points:
1. Latest Trends in Artificial Intelligence:
- Advancements in machine learning algorithms
- Increased adoption of AI in healthcare, finance, and retail sectors
- Rise of conversational AI and virtual assistants

2. Key Pla

I now can give a great answer

Final Answer:

# Unraveling the World of Artificial Intelligence: Trends, Players, and News

Artificial Intelligence (AI) has become an integral part of various industries, transforming the way we interact with technology and conduct business. It is crucial for professionals and enthusiasts to stay informed about the latest trends and developments in AI to navigate the rapidly changing landscape of technology effectively.

In the realm of AI, one of the most notable trends is the continuous improvement of machine learning algorithms. These advancements have led to enhanced predictive capabilities and decision-making processes, driving efficiency and innovation across sectors like healthcare, finance, and retail. Additionally, the growing popularity of conversational AI and virtual assistants is revolutionizing human-computer interactions, making them more seamless and intuitive.

The AI industry is dominated by key players like Google's DeepMind, IBM Wats

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# Unraveling the World of Artificial Intelligence: Trends, Players, and News

Artificial Intelligence (AI) has become an integral part of various industries, transforming the way we interact with technology and conduct business. It is crucial for professionals and enthusiasts to stay informed about the latest trends and developments in AI to navigate the rapidly changing landscape of technology effectively.

In the realm of AI, one of the most notable trends is the continuous improvement of machine learning algorithms. These advancements have led to enhanced predictive capabilities and decision-making processes, driving efficiency and innovation across sectors like healthcare, finance, and retail. Additionally, the growing popularity of conversational AI and virtual assistants is revolutionizing human-computer interactions, making them more seamless and intuitive.

The AI industry is dominated by key players like Google's DeepMind, IBM Watson, Amazon Web Services (AWS), and Microsoft Azure AI, who are at the forefront of AI innovation. These companies are investing significantly in research and development to push the boundaries of artificial intelligence technologies, driving progress and shaping the future of AI applications.

Recent breakthroughs in natural language processing have propelled AI to new heights, empowering machines to comprehend and generate human language accurately. Discussions on ethical considerations in AI, such as bias, privacy, and transparency, are also gaining traction, shaping the ethical framework of AI applications. Furthermore, AI-powered advancements in autonomous vehicles and robotics are transforming transportation and manufacturing industries, paving the way for a more automated and efficient future.

In conclusion, Artificial Intelligence is a dynamic field with vast potential to revolutionize industries and society as a whole. By staying updated on the latest trends, key players, and noteworthy news in AI, individuals can gain valuable insights into the impact of AI on different sectors. As technology advances, it is essential to embrace the possibilities AI offers while considering the ethical implications and societal impact of these advancements. Subscribe to our blog for regular updates on AI trends and news and engage in discussions on the future of artificial intelligence.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [15]:
topic = "Financial Independence Retire Early - FIRE"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Financial Independence Retire Early - FIRE.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Content Plan Document:

Title: Achieving Financial Independence Retire Early - A Comprehensive Guide to FIRE

Introduction:
- Brief overview of the Financial Independence Retire Early (FIRE) movement
- Importance of early retirement and financial independence
- Mention of key players and trends in the FIRE community

Key Points:
1. Understanding the FIRE Movement
- Definition of FIRE and its principles (high savings rate, frugal living, and smart investing)
- Overview of the 4% rule and safe

I now can give a great answer

Final Answer:
# Achieving Financial Independence Retire Early - A Comprehensive Guide to FIRE

## Introduction:
The Financial Independence Retire Early (FIRE) movement has been gaining momentum in recent years, attracting individuals who are eager to break free from traditional work constraints and achieve financial freedom at a young age. The concept of FIRE revolves around the idea of saving a large portion of one's income, adopting a frugal lifestyle, and investing wisely to build a nest egg that can sustain early retirement. This movement has sparked a shift in mindset towards redefining the traditional notion of retirement and prioritizing financial independence.

Key players in the FIRE community, such as Mr. Money Mustache and J.L. Collins, have shared their success stories and practical tips on achieving FIRE. These influencers have inspired a new generation of individuals to take control of their finances, set ambitious goals, and work towards ea

In [16]:
Markdown(result)

# Achieving Financial Independence Retire Early - A Comprehensive Guide to FIRE

## Introduction:
The Financial Independence Retire Early (FIRE) movement has been gaining momentum in recent years, attracting individuals who are eager to break free from traditional work constraints and achieve financial freedom at a young age. The concept of FIRE revolves around the idea of saving a large portion of one's income, adopting a frugal lifestyle, and investing wisely to build a nest egg that can sustain early retirement. This movement has sparked a shift in mindset towards redefining the traditional notion of retirement and prioritizing financial independence.

Key players in the FIRE community, such as Mr. Money Mustache and J.L. Collins, have shared their success stories and practical tips on achieving FIRE. These influencers have inspired a new generation of individuals to take control of their finances, set ambitious goals, and work towards early retirement. The FIRE movement emphasizes the importance of financial literacy, smart investing, and disciplined saving habits to pave the way for a secure and fulfilling future.

## Understanding the FIRE Movement:
At the core of the FIRE movement are three key principles: maintaining a high savings rate, embracing frugal living, and making strategic investments. The 4% rule and safe withdrawal rate are commonly used guidelines in the FIRE community to determine how much one can safely withdraw from their investment portfolio in retirement. By adhering to these principles, individuals can achieve financial independence and have the freedom to pursue their passions without the constraints of traditional employment.

The benefits of achieving FIRE go beyond early retirement; they include increased freedom, flexibility, and reduced financial stress. By prioritizing financial independence, individuals can design their lives on their own terms, pursue meaningful experiences, and create a sense of security for themselves and their loved ones. The FIRE movement offers a roadmap to financial freedom and empowers individuals to take control of their financial future.

## Steps to Achieve FIRE:
To embark on the journey towards FIRE, individuals must first set clear financial goals and create a budget that aligns with their long-term objectives. Increasing income through side hustles or passive income streams can accelerate progress towards financial independence. Adopting sound investing strategies, such as diversification and long-term growth portfolios, can help individuals build wealth over time and secure their financial future.

In addition to saving and investing, building emergency funds and having adequate insurance coverage are crucial components of achieving FIRE. These safety nets protect individuals from unforeseen financial setbacks and provide peace of mind as they work towards early retirement. By following a disciplined approach to financial planning and staying committed to their goals, individuals can make significant strides towards achieving FIRE and realizing their dreams of early retirement.

## Conclusion:
The FIRE movement offers a compelling alternative to traditional retirement planning, emphasizing the importance of financial independence, frugal living, and smart investing. By following the principles of FIRE and taking proactive steps towards early retirement, individuals can create a path to financial freedom and unlock new possibilities for their future. As more individuals embrace the FIRE mindset and work towards their goals, the movement continues to inspire a generation of savers, investors, and dreamers to pursue a life of purpose and fulfillment.

In conclusion, achieving Financial Independence Retire Early is not just a financial goal; it is a mindset shift that empowers individuals to take control of their finances, pursue their passions, and create a life of abundance and fulfillment. By embracing the principles of FIRE and following a strategic roadmap towards early retirement, individuals can set themselves on a path to financial freedom and embark on a journey towards a brighter and more secure future.

I am confident that this blog post on Financial Independence Retire Early - FIRE will provide valuable insights and guidance to readers interested in achieving financial independence and early retirement. By incorporating SEO keywords, engaging storytelling, and practical tips, this article aims to inspire and inform individuals on their journey towards FIRE.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).